# Speaker Recognition Experiment (SpeechBrain)

This notebook tests speaker identification using SpeechBrain's ECAPA-TDNN model.

## Goal
Enable Yumi to recognize the user's voice for personalization/security.

## Approach
1. Extract voice embeddings (192-dim vectors) from audio
2. Store embeddings for known speakers
3. Compare new audio using cosine similarity
4. Threshold: >0.75 = same speaker

## Outcome
**Status**: Working prototype, NOT integrated in v1

### Why not in production
- Adds ~500ms latency per interaction
- Not critical for core waifu experience
- Can be added in v2 as optional feature
- Requires persistent voice database

## Model Details
- **Model**: speechbrain/spkrec-ecapa-voxceleb
- **Size**: ~90MB download
- **Architecture**: ECAPA-TDNN (state-of-the-art speaker ID)
- **Embedding dim**: 192

## Dependencies
```bash
pip install speechbrain torchaudio scikit-learn
```

In [ ]:
%pip install speechbrain
%pip install torchaudio

In [ ]:
import torch
import torchaudio
import numpy as np
from speechbrain.pretrained import EncoderClassifier
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported")

In [ ]:
# Load Speaker Model (~90MB download on first run)
classifier = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    run_opts={"device": "cpu"}
)
print("Speaker model loaded")

In [ ]:
# Generate Voice Embedding (192-dim vector = voice fingerprint)
def get_voice_embedding(audio_path):
    signal, _ = torchaudio.load(audio_path)
    embedding = classifier.encode_batch(signal)
    embedding = embedding.squeeze().detach().numpy()
    return embedding

In [ ]:
# Test embedding extraction
embedding = get_voice_embedding("clean_speech.wav")
print(f"Embedding shape: {embedding.shape}")  # Expected: (192,)

In [ ]:
# Voice database to store known speakers
voice_db = {}

In [ ]:
# Register a voice sample
def register_voice(name, audio_path):
    embedding = get_voice_embedding(audio_path)
    voice_db[name] = embedding
    print(f"{name} registered")

# Example: register_voice("owner", "clean_speech.wav")

In [ ]:
# Identify speaker by comparing cosine similarity
def identify_speaker(audio_path, threshold=0.75):
    new_embedding = get_voice_embedding(audio_path)
    
    best_match = None
    best_score = -1
    
    for name, emb in voice_db.items():
        score = cosine_similarity([new_embedding], [emb])[0][0]
        if score > best_score:
            best_score = score
            best_match = name
    
    return best_match if best_score > threshold else "unknown", best_score

In [ ]:
# Test speaker recognition
speaker, score = identify_speaker("clean_speech.wav")
print(f"Detected: {speaker} (confidence: {score:.2f})")

In [ ]:
# Threshold reference:
# > 0.75  = same speaker
# 0.6-0.75 = maybe same
# < 0.6   = different speaker